In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/deberta-lora-artifacts/__results__.html
/kaggle/input/deberta-lora-artifacts/__huggingface_repos__.json
/kaggle/input/deberta-lora-artifacts/submission.csv
/kaggle/input/deberta-lora-artifacts/__notebook__.ipynb
/kaggle/input/deberta-lora-artifacts/__output__.json
/kaggle/input/deberta-lora-artifacts/custom.css
/kaggle/input/deberta-lora-artifacts/merged_deberta_lora_model/config.json
/kaggle/input/deberta-lora-artifacts/merged_deberta_lora_model/tokenizer.json
/kaggle/input/deberta-lora-artifacts/merged_deberta_lora_model/tokenizer_config.json
/kaggle/input/deberta-lora-artifacts/merged_deberta_lora_model/model.safetensors
/kaggle/input/deberta-lora-artifacts/__results___files/__results___11_0.png
/kaggle/input/deberta-lora-artifacts/__results___files/__results___32_0.png
/kaggle/input/deberta-lora-artifacts/deberta_lora/checkpoint-16206/adapter_model.safetensors
/kaggle/input/deberta-lora-artifacts/deberta_lora/checkpoint-16206/trainer_state.json
/kaggle/input/deberta-l

In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/deberta-lora-artifacts/__results__.html
/kaggle/input/deberta-lora-artifacts/__huggingface_repos__.json
/kaggle/input/deberta-lora-artifacts/submission.csv
/kaggle/input/deberta-lora-artifacts/__notebook__.ipynb
/kaggle/input/deberta-lora-artifacts/__output__.json
/kaggle/input/deberta-lora-artifacts/custom.css
/kaggle/input/deberta-lora-artifacts/merged_deberta_lora_model/config.json
/kaggle/input/deberta-lora-artifacts/merged_deberta_lora_model/tokenizer.json
/kaggle/input/deberta-lora-artifacts/merged_deberta_lora_model/tokenizer_config.json
/kaggle/input/deberta-lora-artifacts/merged_deberta_lora_model/model.safetensors
/kaggle/input/deberta-lora-artifacts/__results___files/__results___11_0.png
/kaggle/input/deberta-lora-artifacts/__results___files/__results___32_0.png
/kaggle/input/deberta-lora-artifacts/deberta_lora/checkpoint-16206/adapter_model.safetensors
/kaggle/input/deberta-lora-artifacts/deberta_lora/checkpoint-16206/trainer_state.json
/kaggle/input/deberta-l

In [3]:
import os, re, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    Trainer, TrainingArguments, DataCollatorWithPadding
)
from torch.utils.data import Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

YOUR_OUTPUT_PATH = "/kaggle/input/deberta-lora-artifacts/merged_deberta_lora_model"

Device: cuda


In [4]:
class CFG:
    train_path = "/kaggle/input/legal-doc-classification/train.csv"
    test_path  = "/kaggle/input/legal-doc-classification/test.csv"
    id_col = "id"; text_col = "text"; label_col = "label"
    max_length = 512
    head_len = 128; tail_len = 384
    preslice_head_words = 400; preslice_tail_words = 1200
    val_size = 0.15
    min_samples_per_class_for_stratify = 2
    eval_bs = 16

train_df_full = pd.read_csv(CFG.train_path)
test_df_full  = pd.read_csv(CFG.test_path)
train_df_full = train_df_full.dropna(subset=[CFG.text_col, CFG.label_col]).reset_index(drop=True)
train_df_full[CFG.label_col] = train_df_full[CFG.label_col].astype(int)
CFG.num_labels = int(train_df_full[CFG.label_col].max()) + 1

def clean_text(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'Page\s*\d+\s*of\s*\d+', '', text, flags=re.I)
    text = re.sub(r'-{3,}', '', text)
    return text.strip()

def preslice_words(text, head_words, tail_words):
    words = text.split()
    if len(words) <= head_words + tail_words:
        return text
    return " ".join(words[:head_words] + words[-tail_words:])

def preprocess(df):
    df = df.copy()
    df[CFG.text_col] = df[CFG.text_col].apply(clean_text)
    df[CFG.text_col] = df[CFG.text_col].apply(
        lambda t: preslice_words(t, CFG.preslice_head_words, CFG.preslice_tail_words)
    )
    return df

train_df_full = preprocess(train_df_full)
test_df_full  = preprocess(test_df_full)
train_df, test_df = train_df_full, test_df_full

def safe_stratified_split(df, label_col, val_size, min_per_class, seed=SEED):
    counts = df[label_col].value_counts()
    splittable = counts[counts >= max(2, min_per_class)].index
    unsplittable = counts[counts < max(2, min_per_class)].index
    df_s = df[df[label_col].isin(splittable)]
    df_u = df[df[label_col].isin(unsplittable)]
    tr, va = train_test_split(df_s, test_size=val_size, stratify=df_s[label_col], random_state=seed)
    tr = pd.concat([tr, df_u]).sample(frac=1, random_state=seed).reset_index(drop=True)
    va = va.reset_index(drop=True)
    return tr, va

train_split, val_split = safe_stratified_split(
    train_df, CFG.label_col, CFG.val_size, CFG.min_samples_per_class_for_stratify
)
print("Train:", train_split.shape, "Val:", val_split.shape)

Train: (43214, 3) Val: (7626, 3)


In [5]:
tokenizer = AutoTokenizer.from_pretrained(YOUR_OUTPUT_PATH)
model = AutoModelForSequenceClassification.from_pretrained(YOUR_OUTPUT_PATH).to(device)

def head_tail_truncate(text, tokenizer, max_length, head_len, tail_len):
    ids = tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"]
    max_body = max_length - 2
    if len(ids) <= max_body:
        body = ids
    else:
        h = min(head_len, max_body)
        t = max_body - h
        body = ids[:h] + ids[-t:]
    input_ids = [tokenizer.cls_token_id] + body + [tokenizer.sep_token_id]
    attn = [1] * len(input_ids)
    pad_len = max_length - len(input_ids)
    if pad_len > 0:
        input_ids += [tokenizer.pad_token_id] * pad_len
        attn += [0] * pad_len
    return input_ids[:max_length], attn[:max_length]

class DocDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts.tolist()
        self.labels = labels.tolist() if labels is not None else None
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        input_ids, attn = head_tail_truncate(self.texts[idx], tokenizer, CFG.max_length, CFG.head_len, CFG.tail_len)
        item = {"input_ids": torch.tensor(input_ids), "attention_mask": torch.tensor(attn)}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

val_ds  = DocDataset(val_split[CFG.text_col], val_split[CFG.label_col])
test_ds = DocDataset(test_df[CFG.text_col])

infer_trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./tmp_infer",
        per_device_eval_batch_size=CFG.eval_bs,
        fp16=torch.cuda.is_available(),
        report_to="none"
    ),
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)

val_logits  = infer_trainer.predict(val_ds).predictions
test_logits = infer_trainer.predict(test_ds).predictions

val_probs_model  = F.softmax(torch.tensor(val_logits), dim=-1).numpy()
test_probs_model = F.softmax(torch.tensor(test_logits), dim=-1).numpy()

print("val_probs_model:", val_probs_model.shape, "test_probs_model:", test_probs_model.shape)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


val_probs_model: (7626, 30) test_probs_model: (12710, 30)


In [6]:
old_submission = pd.read_csv("/kaggle/input/deberta-lora-artifacts/submission.csv")
new_preds_solo = np.argmax(test_probs_model, axis=1)

match_rate = (old_submission[CFG.label_col].values == new_preds_solo).mean()
print(f"Agreement with original submission: {match_rate:.4f}")

Agreement with original submission: 0.9996


In [7]:
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
X_train = tfidf.fit_transform(train_split[CFG.text_col])
X_val   = tfidf.transform(val_split[CFG.text_col])
X_test  = tfidf.transform(test_df[CFG.text_col])

svm = CalibratedClassifierCV(LinearSVC(class_weight="balanced"), cv=3)
svm.fit(X_train, train_split[CFG.label_col])

val_probs_tfidf  = svm.predict_proba(X_val)
test_probs_tfidf = svm.predict_proba(X_test)

val_labels = val_split[CFG.label_col].values
best_w, best_acc = 0, 0
for w in np.arange(0, 1.05, 0.05):
    blend = w * val_probs_model + (1 - w) * val_probs_tfidf
    acc = accuracy_score(val_labels, np.argmax(blend, axis=1))
    if acc > best_acc:
        best_acc, best_w = acc, w

print(f"Solo model val accuracy : {accuracy_score(val_labels, np.argmax(val_probs_model, axis=1)):.5f}")
print(f"Solo TF-IDF val accuracy: {accuracy_score(val_labels, np.argmax(val_probs_tfidf, axis=1)):.5f}")
print(f"Best blend weight (model): {best_w:.2f} -> val accuracy {best_acc:.5f}")

Solo model val accuracy : 0.66627
Solo TF-IDF val accuracy: 0.72568
Best blend weight (model): 0.25 -> val accuracy 0.72777


In [8]:
final_blend = best_w * test_probs_model + (1 - best_w) * test_probs_tfidf
test_preds = np.argmax(final_blend, axis=1)

submission = pd.DataFrame({'ID': test_df['id'], CFG.label_col: test_preds})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
submission.head()

Saved submission.csv


,ID,label
0,0,7
1,1,2
2,2,3
3,3,11
4,4,0
